### RADIO CLEANING ###

In [1]:
import pandas as pd
import numpy as np
import os

# ============================================================
# 1. CONFIGURATION & PATHS
# ============================================================
file_radio_input = "Datasets/df_radio_2022_2025.csv"
file_urgences = "Datasets/df_ioa_vital_med_adm_tabular.csv"
output_radio_clean = "Datasets/df_radio_clean.csv"  # <-- L'étape intermédiaire
output_final_merge = "Datasets/df_adm_pv_ioa_med_radio_tabular.csv"

mapping_categories = {
    'ultrasound': ['Echographie'],
    'ct_scan': ['Tomodensitométrie'],
    'xray': ['Radiologie conventionnelle'],
    'mri': ['IRM'],
    'radio_interventional': ['Radiologie Interventionnelle'],
    'nuclear_medicine': ['Médecine nucléaire', 'Tep Scan']
}

EXAM_MAP = {
    # --- CT SCANS (TDM) ---
    'TDM ALERTE AVC':                           'ct stroke protocol',
    'TDM ANGIO-TDM DES TSA /POLYGONE DE WILLIS': 'ct angiography neck and circle of willis',
    'TDM ANGIOSCAN CORONAIRES':                 'ct coronary angiography',
    'TDM COLOSCAN':                             'ct colonoscopy',
    'TDM CRANE-ABDOMEN-PELVIS':                 'ct brain abdomen pelvis',
    'TDM CRANE-THORAX-ABDOPELVIS':              'ct brain chest abdomen pelvis',
    'TDM LARYNX':                               'ct larynx',
    'TDM ORBITAIRE':                            'ct orbits',
    'TDM RACHIS + SACCORADICULO':               'ct spine sacroradiculography',
    'TDM RACHIS DORSAL / THORACIQUE':           'ct thoracic spine',
    'TDM THYROIDE/PARATHYROIDE':                'ct thyroid parathyroid',
    'TDM TSA / COEUR':                          'ct neck vessels heart',
    'TDM CRANE/CEREBRAL': 'ct head / brain',
    'TDM TRAUMA CRANE': 'ct head trauma',
    'TDM CRANE-THORAX-ABDOMEN-PELVIS': 'ct head neck chest abdomen pelvis',
    'TDM THORAX-ABDO-PELVIS': 'ct chest abdomen pelvis',
    'TDM ABDO-PELVIS': 'ct abdomen pelvis',
    'TDM ABDOMINAL': 'ct abdomen',
    'TDM THORAX': 'ct chest',
    'TDM ANGIOSCAN CEREBRAL': 'ct angiography brain',
    'TDM ANGIOSCAN THORAX': 'ct angiography chest pulmonary embolism',
    'TDM RACHIS LOMBAIRE': 'ct lumbar spine',
    'TDM RACHIS CERVICAL': 'ct cervical spine',
    'TDM SINUS': 'ct sinuses',
    'TDM ROCHERS': 'ct temporal bone',
    'TDM PERFUSION': 'ct perfusion brain',
    'TDM ANGIOSCAN ABDOMEN +/- PELVIS': 'ct angiography abdomen pelvis',
    'TDM ANGIOSCAN CEREBRAL+ CERVICAL': 'ct angiography brain neck',
    'TDM ANGIOSCAN CERVICAL': 'ct angiography neck',
    'TDM ANGIOSCAN MEMBRE INF': 'ct angiography lower limb',
    'TDM ANGIOSCAN MEMBRE SUP': 'ct angiography upper limb',
    'TDM ANGIOSCAN THORAX + ABDO +/- PELVIS': 'ct angiography chest abdomen pelvis',
    'TDM AORTE ABDOMINALE': 'ct abdominal aorta',
    'TDM AORTE COMPLETE': 'ct whole aorta',
    'TDM AORTE THORACIQUE': 'ct thoracic aorta',
    'TDM BASSIN': 'ct pelvic bone',
    'TDM CAVUM': 'ct cavum',
    'TDM COU': 'ct neck',
    'TDM COU+THORAX-ABDO-PELVIS': 'ct neck chest abdomen pelvis',
    'TDM CRANE + COU': 'ct brain neck',
    'TDM CRANE DOSIMETRIE': 'ct brain dosimetry',
    'TDM CRANE-THORAX': 'ct brain chest',
    'TDM CRANE-THORAX-ABDOMEN': 'ct brain chest abdomen',
    'TDM CRANE/MASSIF FACIAL': 'ct brain facial bones',
    'TDM DRAINAGE CRANE/MASSIF FACIAL': 'ct drainage brain facial bones',
    'TDM INFILTRATION CERVICALE': 'ct guided cervical infiltration',
    'TDM MEMBRE INF D': 'ct right lower limb',
    'TDM MEMBRE INF G': 'ct left lower limb',
    'TDM MEMBRE SUP D': 'ct right upper limb',
    'TDM MEMBRE SUP G': 'ct left upper limb',
    'TDM PELVIS': 'ct pelvis',
    'TDM RACHIS DORSAL': 'ct thoracic spine',
    'TDM RACHIS ENTIER': 'ct whole spine',
    'TDM RACHIS LOMBAIRE AVEC DISCO': 'ct lumbar spine with discography',
    'TDM RACHIS PLUSIEURS SEGMENTS': 'ct multi segment spine',
    'TDM REINS': 'ct kidneys',
    'TDM THORAX + COU': 'ct chest neck',
    "TENTATIVE D'EXAMEN TDM": "attempted ct scan",

    # --- MRI (IRM) ---
    'ANGIO IRM ABDO/PELVIS':            'mri angiography abdomen pelvis',
    'ANGIO IRM MBRE INF G':             'mri angiography left lower limb',
    'IRM ALERTE AIT  - CLINIQUE AIT':   'mri tia protocol',
    'IRM AORTE':                        'mri aorta',
    'IRM CORPS ENTIER':                 'mri whole body',
    'IRM COU':                          'mri neck',
    'IRM COUDE G':                      'mri left elbow',
    'IRM CRANE/ENCEPHALE+MOELLE':       'mri brain and spinal cord',
    'IRM CRANE/OS':                     'mri brain skull',
    'IRM FOIE':                         'mri liver',
    'IRM GENOU D':                      'mri right knee',
    'IRM GENOU G':                      'mri left knee',
    'IRM MBRE INF G':                   'mri left lower limb',
    'IRM MBRE SUP G':                   'mri left upper limb',
    'IRM TESTICULAIRE':                 'mri scrotum',
    'IRM VESSIE':                       'mri bladder',
    'IRM VOIES BILIAIRES':              'mri biliary tract',
    'BILI IRM+FOIE':                    'mrcp liver',
    'BILI IRM+PANCREAS':                'mrcp pancreas',
    'IRM ALERTE AVC': 'mri stroke protocol',
    'IRM CRANE/ENCEPHALE': 'mri brain',
    'IRM CRANE DIFF-PERF': 'mri brain diffusion perfusion',
    'ANGIO IRM CERVICO CEREBRAL': 'mri angiography neck brain',
    'IRM RACHIS LOMBAIRE': 'mri lumbar spine',
    'IRM RACHIS CERVICAL': 'mri cervical spine',
    'IRM MEDULLAIRE': 'mri spinal cord',
    'IRM ABDOMEN': 'mri abdomen',
    'IRM PELVIS': 'mri pelvis',
    'IRM CAI': 'mri internal auditory canal',
    'ANGIO IRM CEREBRALE': 'mri angiography brain',
    'ANGIO IRM CERVICALE': 'mri angiography neck',
    'ANGIO IRM MBRE INF D': 'mri angiography right lower limb',
    'ANGIO IRM RACHIS': 'mri angiography spine',
    'IRM BASSIN': 'mri pelvis',
    'IRM CHEVILLE/PIED G': 'mri left ankle foot',
    'IRM COUDE D': 'mri right elbow',
    'IRM CRANE DIFF-PERF + ANGIO IRM': 'mri brain diffusion perfusion and angiography',
    'IRM CRANE/ENCEPHALE + MOELLE': 'mri brain and spinal cord',
    'IRM CRANE/MASSIF FACIAL': 'mri brain facial bones',
    'IRM HYPOPHYSE': 'mri pituitary gland',
    'IRM MBRE INF D': 'mri right lower limb',
    'IRM OFSEP MOELLE': 'mri spinal cord multiple sclerosis protocol',
    'IRM ORBITES': 'mri orbits',
    'IRM RACHIS COMPLET': 'mri whole spine',
    'IRM RACHIS DORSAL': 'mri thoracic spine',
    "TENTATIVE D'EXAMEN IRM": "attempted mri",

    # --- X-RAY (RADIO) ---
    'RADIO  THORAX + ASP':                      'xray chest and abdomen supine',
    'RADIO ATLAS AXIS':                         'xray atlas axis',
    'RADIO AVANT BRAS D + MAIN D':              'xray right forearm and hand',
    'RADIO BILAN OSSEUX PEDIATRIQUE':           'xray pediatric bone survey',
    'RADIO CALCANEUM D  F+P':                   'xray right calcaneus',
    'RADIO CALCANEUM G  F+P':                   'xray left calcaneus',
    'RADIO CHARNIERE CERVICO-DORSALE F+P':      'xray cervicothoracic junction',
    'RADIO CHARNIERE LOMBO-SACREE F+P':         'xray lumbosacral junction',
    'RADIO CHEVILLES COMPARATIF D+G':           'xray bilateral ankles comparative',
    'RADIO CONTRÔLE DERIVATION':                'xray shunt control',
    'RADIO CRANE BLONDEAU':                     'xray skull blondeau view',
    'RADIO CRANE FACE BASSE':                   'xray skull submentovertex',
    'RADIO EPAULE G PROFIL':                    'xray left shoulder lateral',
    'RADIO GENOU D  F+P EN CHARGE':             'xray right knee weight bearing',
    'RADIO GENOU G  F+P EN CHARGE':             'xray left knee weight bearing',
    'RADIO GENOUX COMPARATIF D+G':              'xray bilateral knees comparative',
    'RADIO GRIL COSTAL BILATÉRAL':              'xray bilateral ribs',
    'RADIO MEMBRE INFERIEUR + BASSIN':          'xray lower limb and pelvis',
    'RADIO MEMBRE SUPERIEUR D FACE':            'xray right upper limb ap',
    'RADIO MEMBRE SUPERIEUR G FACE':            'xray left upper limb ap',
    'RADIO PIED D  F+P EN CHARGE':              'xray right foot weight bearing',
    'RADIO PIED D F + P DYNAMIQUES':            'xray right foot dynamic views',
    'RADIO PIED G  F+P EN CHARGE':              'xray left foot weight bearing',
    'RADIO PROTOCOLE IOA':                      'xray ioa protocol',
    'RADIO RACHIS CERVICAL  DYNAMIQUES':        'xray cervical spine dynamic views',
    'RADIO RACHIS CERVICAL  F+P+ ATLAS/AXIS':   'xray cervical spine with atlas axis',
    'RADIO RACHIS CERVICAL  F+P+DYNAMIQUES':    'xray cervical spine with dynamic views',
    'RADIO RACHIS CERVICAL + LOMBAIRE F+P':     'xray cervical and lumbar spine',
    'RADIO RACHIS LOMBAIRE  DESEZE':            'xray lumbar spine deseze view',
    'RADIO SCAPHOIDE D 4I':                     'xray right scaphoid 4 views',
    'RADIO SCAPHOIDE G 4I':                     'xray left scaphoid 4 views',
    'RADIO STERNUM F+P':                        'xray sternum',
    'RADIO TELECRANE':                          'xray skull telecranium',
    'RADIO THORAX + CAVUM':                     'xray chest and nasopharynx',
    'RADIO THORAX + CRANE':                     'xray chest and skull',
    'RADIO THORAX INSPI EXPI':                  'xray chest inspiration expiration',
    'RADIO THORAX': 'xray chest',
    'RADIO THORAX AU LIT': 'xray chest bedside',
    'RADIO ABDOMEN SANS PREPARATION': 'xray abdomen supine',
    'RADIO ABDOMEN SANS PREPARATION AU LIT': 'xray abdomen supine bedside',
    'RADIO BASSIN': 'xray pelvis',
    'RADIO RACHIS LOMBAIRE F+P': 'xray lumbar spine',
    'RADIO EPAULE D  F+P': 'xray right shoulder',
    'RADIO EPAULE G  F+P': 'xray left shoulder',
    'RADIO GENOU D F+P': 'xray right knee',
    'RADIO CHEVILLE D F+P': 'xray right ankle',
    'RADIO POIGNET D  F+P': 'xray right wrist',
    'RADIO MAIN D F+ 3/4': 'xray right hand',
    'RADIO ACROMIO-CLAVICULAIRE D': 'xray right acromioclavicular joint',
    'RADIO ACROMIO-CLAVICULAIRE G': 'xray left acromioclavicular joint',
    'RADIO ADULTES BILAN': 'xray adult survey',
    'RADIO AVANT BRAS G + MAIN G': 'xray left forearm and hand',
    'RADIO AVANT BRAS G + MAIN D': 'xray right forearm and hand',
    'RADIO AVANT-BRAS D  F+P': 'xray right forearm',
    'RADIO AVANT-BRAS D + COUDE D  F+P': 'xray right forearm and elbow',
    'RADIO AVANT-BRAS G + COUDE G  F+P': 'xray left forearm and elbow',
    'RADIO AVANT-BRAS G  F+P': 'xray left forearm',
    'RADIO BASSIN  F': 'xray pelvis ap',
    'RADIO BASSIN + HANCHE D': 'xray pelvis and right hip',
    'RADIO BASSIN + HANCHE G': 'xray pelvis and left hip',
    'RADIO BASSIN + HANCHES': 'xray pelvis and hips',
    'RADIO BILAN EXTRAVASATION': 'xray extravasation survey',
    'RADIO BILAN URGENCE': 'xray emergency survey',
    'RADIO CHEVILLE D + PIED D': 'xray right ankle and foot',
    'RADIO CHEVILLE G  F+P': 'xray left ankle',
    'RADIO CHEVILLE D  F+P': 'xray right ankle',
    'RADIO CHEVILLE G + PIED G': 'xray left ankle and foot',
    'RADIO CLAVICULE D  F': 'xray right clavicle ap',
    'RADIO CLAVICULE G  F': 'xray left clavicle ap',
    'RADIO CONTROLE DERIVATION': 'xray shunt control',
    'RADIO COUDE D  F+P': 'xray right elbow',
    'RADIO COUDE G  F+P': 'xray left elbow',
    'RADIO CRANE': 'xray skull',
    'RADIO DOIGT F+P': 'xray finger',
    'RADIO DOIGT F+P+ 3/4': 'xray finger oblique',
    'RADIO EPAULE G  F+P+ ROTATIONS': 'xray left shoulder with rotations',
    'RADIO EPAULE G FACE': 'xray left shoulder ap',
    'RADIO EPAULE D  F+P+ ROTATIONS': 'xray right shoulder with rotations',
    'RADIO EPAULE D FACE': 'xray right shoulder ap',
    'RADIO FEMUR D  F+P': 'xray right femur',
    'RADIO FEMUR G  F+P': 'xray left femur',
    'RADIO GENOU D DFP': 'xray right knee tangential view',
    'RADIO GENOU G DFP': 'xray left knee tangential view',
    'RADIO GENOU G  F+P': 'xray left knee',
    'RADIO GENOU D  F+P': 'xray right knee',
    'RADIO GRIL COSTAL BILAT': 'xray bilateral ribs',
    'RADIO GRIL COSTAL D': 'xray right ribs',
    'RADIO GRIL COSTAL G': 'xray left ribs',
    'RADIO HANCHE D  F+P': 'xray right hip',
    'RADIO HANCHE G  F+P': 'xray left hip',
    'RADIO HUMERUS D  F+P': 'xray right humerus',
    'RADIO HUMERUS G  F+P': 'xray left humerus',
    'RADIO JAMBE D  F+P': 'xray right lower leg',
    'RADIO JAMBE D + PIED D': 'xray right lower leg and foot',
    'RADIO JAMBE G  F+P': 'xray left lower leg',
    'RADIO JAMBE G + PIED G': 'xray left lower leg and foot',
    'RADIO MAIN D + POIGNET D F+P': 'xray right hand and wrist',
    'RADIO MAIN G  F+ 3/4': 'xray left hand oblique',
    'RADIO MAIN D  F+ 3/4': 'xray right hand oblique',
    'RADIO MAIN G + POIGNET G F+P': 'xray left hand and wrist',
    'RADIO MAINS D ET G': 'xray bilateral hands',
    'RADIO MEMBRE INF D + HANCHE D': 'xray right lower limb and hip',
    'RADIO MEMBRE INF G + HANCHE G': 'xray left lower limb and hip',
    'RADIO MEMBRE INFERIEUR  D  F+P': 'xray right lower limb',
    'RADIO MEMBRE INFERIEUR  G  F+P': 'xray left lower limb',
    'RADIO MEMBRE SUPERIEUR  D  F+P': 'xray right upper limb',
    'RADIO MEMBRE SUPERIEUR  G  F+P': 'xray left upper limb',
    'RADIO MEMBRES INFERIEURS D ET G F': 'xray bilateral lower limbs ap',
    'RADIO OMOPLATE G  F+P': 'xray left scapula',
    'RADIO OMOPLATE D  F+P': 'xray right scapula',
    'RADIO OS PROPRES DU NEZ': 'xray nasal bones',
    'RADIO PANORAMIQUE DENTAIRE': 'xray dental panoramic',
    'RADIO PIED D  F+ 3/4': 'xray right foot oblique',
    'RADIO PIED G  F+ 3/4': 'xray left foot oblique',
    'RADIO PIEDS D ET G  F': 'xray bilateral feet ap',
    'RADIO POIGNET D + AVANT-BRAS D  F+P': 'xray right wrist and forearm',
    'RADIO POIGNET D ETG COMPARATIVE': 'xray bilateral wrists comparative',
    'RADIO POIGNET G  F+P': 'xray left wrist',
    'RADIO POIGNET G + AVANT-BRAS G  F+P': 'xray left wrist and forearm',
    'RADIO RACHIS + BASSIN FACE': 'xray spine and pelvis ap',
    'RADIO RACHIS CERVICAL  F+P': 'xray cervical spine',
    'RADIO RACHIS CERVICAL + DORSAL F+P': 'xray cervical and thoracic spine',
    'RADIO RACHIS COMPLET F+P': 'xray whole spine',
    'RADIO RACHIS COMPLET SEGMENTAIRE F+P': 'xray whole spine segmental',
    'RADIO RACHIS DORSAL  F+P': 'xray thoracic spine',
    'RADIO RACHIS DORSAL + LOMBAIRE F+P': 'xray thoracic and lumbar spine',
    'RADIO RACHIS LOMBAIRE  F+P': 'xray lumbar spine',
    'RADIO SACRUM COCCYX  F+P': 'xray sacrum coccyx',
    'RADIO THORAX + ASP': 'xray chest and abdomen supine',
    "TENTATIVE D'EXAMEN RADIO": "attempted xray",

    # --- ULTRASOUND (ECHO) ---
    'ECHO BIOPSIE MEMBRE SUPERIEUR':            'upper limb ultrasound guided biopsy',
    'ECHO DOPPLER HEPATIQUE':                   'hepatic doppler ultrasound',
    'ECHO DOPPLER RENAL SONOVUE':               'renal doppler ultrasound with contrast',
    'ECHO DOPPLER VEINE CAVE':                  'vena cava doppler ultrasound',
    'ECHO DRAINAGE COLLECTION MBRE SUP':        'upper limb ultrasound guided drainage',
    'ECHO OBSTETRICALE':                        'obstetric ultrasound',
    'ECHO PARTIES MOLLES CRANE':                'skull soft tissue ultrasound',
    'ECHO PARTIES MOLLES RACHIS':               'spine soft tissue ultrasound',
    'ECHO PARTIES MOLLES THORAX':               'chest soft tissue ultrasound',
    'ECHO PONCTION / BIOPSIE MBRE SUP':         'upper limb ultrasound guided biopsy puncture',
    'ECHO PONCTION MEMBRE SUPERIEUR':           'upper limb ultrasound guided puncture',
    'ECHO THORACIQUE':                          'thoracic ultrasound',
    'ECHOGRAPHIE MAMMAIRE':                     'breast ultrasound',
    "TENTATIVE D'EXAMEN ECHOGRAPHIE":           'attempted ultrasound',
    'ECHO ABDOMINALE': 'abdominal ultrasound',
    'ECHO DOPPLER VAISSEAUX DU COU': 'carotid doppler ultrasound',
    'ECHO DOPPLER VEINEUX MBRE INF': 'venous doppler ultrasound lower limb ',
    'ECHO DOPPLER ARTERIEL MBRE INF': 'arterial doppler ultrasound lower limb',
    'ECHO REINS-PELVIS': 'renal and bladder ultrasound',
    'ECHO TESTICULAIRE': 'scrotal ultrasound',
    'ECHO ABDO-PELVIS': 'abdomen pelvis ultrasound',
    'ECHO ARTICULATION MBRE INF': 'lower limb joint ultrasound',
    'ECHO ARTICULATION MBRE SUP': 'upper limb joint ultrasound',
    'ECHO DOPPLER ABDOMEN': 'abdomen doppler ultrasound',
    'ECHO DOPPLER ABDOMEN PELVIS': 'abdomen pelvis doppler ultrasound',
    'ECHO DOPPLER ABORD DE DIALYSE': 'dialysis access doppler ultrasound',
    'ECHO DOPPLER ANGIOME': 'angioma doppler ultrasound',
    'ECHO DOPPLER ARTERIEL MBRE SUP': 'arterial doppler ultrasound upper limb',
    'ECHO DOPPLER GREFFON RENAL': 'renal transplant doppler ultrasound',
    'ECHO DOPPLER PENIEN': 'penile doppler ultrasound',
    'ECHO DOPPLER RENAL': 'renal doppler ultrasound',
    'ECHO DOPPLER TESTICULAIRE': 'scrotal doppler ultrasound',
    'ECHO DOPPLER TSA PRE-GREFFE': 'carotid doppler ultrasound pre transplant',
    'ECHO DOPPLER VAISSEAUX PELVIENS': 'pelvic vessel doppler ultrasound',
    'ECHO DOPPLER VEINEUX MBRE SUP': 'venous doppler ultrasound upper limb',
    'ECHO FOIE-VOIES BILIAIRES': 'liver and biliary tract ultrasound',
    'ECHO HANCHE': 'hip ultrasound',
    'ECHO JAMBE GAUCHE': 'left leg ultrasound',
    'ECHO PARTIES MOLLES ABDOMEN': 'abdomen soft tissue ultrasound',
    'ECHO PARTIES MOLLES COU': 'neck soft tissue ultrasound',
    'ECHO PARTIES MOLLES MBRE INF': 'lower limb soft tissue ultrasound',
    'ECHO PARTIES MOLLES MBRE SUP': 'upper limb soft tissue ultrasound',
    'ECHO PELVIS': 'pelvic ultrasound',
    'ECHO PONCTION / BIOPSIE MBRE INF': 'lower limb ultrasound guided biopsy',
    'ECHO PONCTION MEMBRE INFERIEUR': 'lower limb ultrasound guided puncture',
    'ECHO RENALE-SURRENALE': 'renal adrenal ultrasound',
    'ECHO TESTICULAIRE SONOVUE': 'scrotal ultrasound with contrast',
    'ECHO VESSIE': 'bladder ultrasound',

    # --- INTERVENTIONAL & SPECIAL ---
    'THROMBECTOMIE/AVC': 'mechanical thrombectomy stroke',
    'POSE DE PICC LINE': 'picc line insertion',
    'POSE DE MID LINE': 'midline catheter insertion',
    'PONCTION LOMBAIRE': 'lumbar puncture',
    'NEPHROSTOMIE': 'percutaneous nephrostomy',
    'DRAINAGE PELVIS': 'pelvic drainage',
    'SCINTI POUMONS VENT/PERF': 'v/q lung scintigraphy pulmonary embolism',
    'SCINTI POUMONS VENT/PERF AU KRYPTON': 'v/q lung scintigraphy pulmonary embolism krypton',
    'CHANGEMENT SONDE DE GASTROSTOMIE': 'gastrostomy tube replacement',
    'DETHROMBOSE ANSE FAV': 'av fistula thrombectomy',
    'DRAINAGE HEPATIQUE': 'hepatic drainage',
    'PONCTION / BIOPSIE GENITO URINAIRE': 'genitourinary guided biopsy',
    'PELVIC DRAINAGE': 'pelvic drainage',
    'ANGIOPLASTIE ABDOMINALE':                          'abdominal angioplasty',
    'ARTERIO ARTERES ILIAQUES':                         'iliac arteries arteriography',
    'ARTERIO ARTERES RENALES':                          'renal arteries arteriography',
    'ARTERIOGRAPHIE CEREBRALE':                         'cerebral arteriography',
    'CIMENTOPLASTIE':                                   'cementoplasty',
    'DRAINAGE ABCES/COLLECTION':                        'abscess collection drainage',
    'DRAINAGE ABDOMINAL':                               'abdominal drainage',
    'DRAINAGE BILIAIRE':                                'biliary drainage',
    'EMBOLISATION  ANEVRYSME INTRA-CRANIEN':            'intracranial aneurysm embolization',
    'EMBOLISATION ANEVRYSME MEMBRE INF':                'lower limb aneurysm embolization',
    'EMBOLISATION ANEVRYSME SPLENIQUE':                 'splenic aneurysm embolization',
    'EMBOLISATION ARTERE ABDO (HEMATOME,PSOAS,DIVERTI)': 'abdominal artery embolization',
    'EMBOLISATION ARTERE GASTRIQUE':                    'gastric artery embolization',
    'EMBOLISATION ARTERE HEPATIQUE':                    'hepatic artery embolization',
    'EMBOLISATION ARTERE PELVIENNE HOMME':              'male pelvic artery embolization',
    'EMBOLISATION ARTERE RENALE':                       'renal artery embolization',
    'EMBOLISATION ARTERE SPLENIQUE':                    'splenic artery embolization',
    'EMBOLISATION GASTRO-DUODENALE':                    'gastroduodenal artery embolization',
    'INFILTRATION LOMBAIRE':                            'lumbar infiltration',
    'INJECTION SPINRAZA':                               'spinraza intrathecal injection',
    'PONCTION / BIOPSIE DISCALE / VERTEBRALE':          'vertebral disc biopsy puncture',
    'PONCTION / BIOPSIE RENALE':                        'renal biopsy puncture',
    'POSE DE SONDE GASTROSTOMIE':                       'gastrostomy tube insertion',
    'POSE DE VOIE VEINEUSE PERIPHERIQUE SOUS ECHO':     'peripheral iv line insertion ultrasound guided',


    #    --- NUCLEAR MEDICINE (SCINTI / TEP) ---
    'SCINTI DATSCAN DEMI DOSE':             'datscan scintigraphy half dose',
    'SCINTI DATSCAN IMAGES':                'datscan scintigraphy',
    'SCINTI POLYNUCLEAIRES TC 99M':         'tc99m white blood cell scintigraphy',
    'SCINTI POUMONS IMAGES PERFUSION':      'lung perfusion scintigraphy',
    'SCINTI POUMONS PERFUSION':             'lung perfusion scintigraphy',
    'SCINTI SHUNT PULMONAIRE':              'pulmonary shunt scintigraphy',
    'TEP AUTRE':                            'pet scan other',
    'TEP AVEC IV':                          'pet scan with iv contrast',
    'TEP GYNECOLOGIE':                      'pet scan gynaecology',
    'TEP ORL':                              'pet scan ent',
    'TEP PULMONAIRE':                       'pet scan pulmonary',
}


# ============================================================
# 2. FUNCTIONS
# ============================================================
def clean_row_senior_priority(row):
    seen_labels = set()
    for i in range(len(row) - 1, -1, -1):
        val = str(row.iloc[i]).strip().upper()
        if val in ['NAN', '', 'NONE', 'NAT', 'NULL']:
            row.iloc[i] = np.nan
            continue
        if val in seen_labels:
            row.iloc[i] = np.nan
        else:
            seen_labels.add(val)
    return row

def shift_row_left(row):
    real_exams = [v for v in row if pd.notna(v)]
    return pd.Series(real_exams + [np.nan] * (len(row) - len(real_exams)), index=row.index)



def translate_exam(val):
    # 1. Si la case est vide ou contient un texte type "nan"
    if pd.isna(val) or str(val).upper() in ['NONE', 'NAN', '', 'NULL']:
        return np.nan

    # 2. On prépare la version pour la recherche (nettoyée et en majuscules)
    val_str = str(val).strip()
    val_upper = val_str.upper()

    # 3. LE MAPPING (Priorité au dictionnaire)
    if val_upper in EXAM_MAP:
        return EXAM_MAP[val_upper].lower()

    # 4. LE ELSE (Le fallback)
    # Si on ne connaît pas la traduction, on garde le texte original
    # mais on force les minuscules pour ton modèle d'IA
    else:
        return val_str.lower()

# ============================================================
# 3. INTERMEDIARY STEP: CLEANING AND SAVING RADIO DATA
# ============================================================
print("Step 1: Cleaning Radio Data...")
df_radio = pd.read_csv(file_radio_input, dtype={'nda': str}, low_memory=False)
df_radio['nda'] = df_radio['nda'].str.strip()

exam_keywords = [item for sublist in mapping_categories.values() for item in sublist]
label_cols = [c for c in df_radio.columns if any(k in c for k in exam_keywords)
              and not any(ex in c for ex in ['_date', '_report', 'has_', 'count_', 'total_'])]

all_new_labels = []
for short_name, keywords in mapping_categories.items():
    original_cols = [c for c in label_cols if any(k in c for k in keywords)]
    if not original_cols: continue

    df_radio[original_cols] = df_radio[original_cols].apply(clean_row_senior_priority, axis=1)
    df_radio[original_cols] = df_radio[original_cols].apply(shift_row_left, axis=1)

    rename_map = {old: f"{short_name}_{i+1}" for i, old in enumerate(original_cols)}
    df_radio.rename(columns=rename_map, inplace=True)
    all_new_labels.extend(rename_map.values())

    df_radio[f'has_{short_name}'] = df_radio[list(rename_map.values())].notna().any(axis=1).astype(int)
    df_radio[f'count_{short_name}'] = df_radio[list(rename_map.values())].notna().sum(axis=1)

# TRANSLATING LABELS TO ENGLISH
for col in all_new_labels:
    df_radio[col] = df_radio[col].apply(translate_exam)

# SAVING INTERMEDIATE CLEANED RADIO DATA
df_radio.to_csv(output_radio_clean, index=False)
print(f"Intermediate file saved: {output_radio_clean}")

# ============================================================
# 4. FINAL STEP: MERGE WITH ED DATA
# ============================================================
print("🔗 Step 2: Merging with ED data...")
df_urgences = pd.read_csv(file_urgences, dtype={'nda': str})
df_urgences['nda'] = df_urgences['nda'].str.strip()

# Selecting columns for merge
has_cols = [c for c in df_radio.columns if c.startswith('has_')]
count_cols = [c for c in df_radio.columns if c.startswith('count_')]
final_radio_cols = ['nda'] + has_cols + count_cols + all_new_labels

df_final = pd.merge(df_urgences, df_radio[final_radio_cols], on='nda', how='left')

# Post-merge cleaning
df_final[has_cols + count_cols] = df_final[has_cols + count_cols].fillna(0).astype(int)
df_final['imaging_exam_count'] = df_final[count_cols].sum(axis=1)
df_final['has_imaging_any'] = (df_final['imaging_exam_count'] > 0).astype(int)

# Removing columns that are entirely empty (100% NaN)
existing_exam_cols = [c for c in all_new_labels if c in df_final.columns]
empty_cols = [c for c in existing_exam_cols if df_final[c].isna().all()]
df_final.drop(columns=empty_cols, inplace=True)

df_final.to_csv(output_final_merge, index=False)
print(f"Final merge complete: {output_final_merge}")


Step 1: Cleaning Radio Data...


Intermediate file saved: Datasets/df_radio_clean.csv
🔗 Step 2: Merging with ED data...


/tmp/ipykernel_617750/2922315300.py:447: DtypeWarning:

Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.



Final merge complete: Datasets/df_adm_pv_ioa_med_radio_tabular.csv


In [2]:
# --- Vue globale ---
all_untranslated = set()
for col in all_new_labels:
    if col in df_radio.columns:
        vals = df_radio[col].dropna().unique()
        mapped_values = set(v.lower() for v in EXAM_MAP.values())
        for v in vals:
            if str(v).lower() not in mapped_values:
                all_untranslated.add(str(v).upper())

print(f"\n=== {len(all_untranslated)} UNTRANSLATED EXAMS ===")
for v in sorted(all_untranslated):
    print(f"  '{v}'")


=== 0 UNTRANSLATED EXAMS ===


In [3]:
# import pandas as pd
# import numpy as np
# import os
#
# # ============================================================
# # 1. CONFIGURATION & PATHS
# # ============================================================
# file_radio_input = "df_radio_subset22pel.csv"
# file_urgences = "df_ioa_vital_med_adm_pel_2022_tabular.csv"
# output_radio_clean = "df_radio_clean_subsetpel22.csv"
# output_final_merge = "df_adm_pv_ioa_med_radio_pel22_tabular.csv"
#
# mapping_categories = {
#     'ultrasound': ['Echographie'],
#     'ct_scan': ['Tomodensitométrie'],
#     'xray': ['Radiologie conventionnelle'],
#     'mri': ['IRM'],
#     'radio_interventional': ['Radiologie Interventionnelle'],
#     'nuclear_medicine': ['Médecine nucléaire', 'Tep Scan']
# }
#
# # ============================================================
# # 2. LOAD & INITIAL AUDIT
# # ============================================================
# df_radio = pd.read_csv(file_radio_input, dtype={'nda': str}, low_memory=False)
# df_radio['nda'] = df_radio['nda'].str.strip()
#
# print(f"📊 Initial Radio observations: {len(df_radio)}")
#
# # Identify label columns (avoiding dates and reports)
# exam_keywords = [item for sublist in mapping_categories.values() for item in sublist]
# label_cols = [c for c in df_radio.columns if any(k in c for k in exam_keywords)
#               and not any(ex in c for ex in ['_date', '_report', 'has_', 'count_', 'total_'])]
#
# # ============================================================
# # 3. CLEANING & SHIFT FUNCTIONS
# # ============================================================
# def clean_row_senior_priority(row):
#     """
#     Ensures unique exam labels per patient. Priority to Senior:
#     keeps the last occurrence and sets previous duplicates to NaN.
#     """
#     seen_labels = set()
#     for i in range(len(row) - 1, -1, -1):
#         val = str(row.iloc[i]).strip().upper()
#         if val in ['NAN', '', 'NONE', 'NAT', 'NULL']:
#             row.iloc[i] = np.nan
#             continue
#         if val in seen_labels:
#             row.iloc[i] = np.nan
#         else:
#             seen_labels.add(val)
#     return row
#
# def shift_row_left(row):
#     """Pushes all valid exam labels to the left to fill gaps."""
#     real_exams = [v for v in row if pd.notna(v)]
#     return pd.Series(real_exams + [np.nan] * (len(row) - len(real_exams)), index=row.index)
#
# # ============================================================
# # 4. PROCESSING: HARMONIZE, CLEAN, AND COUNT
# # ============================================================
# print("🛠️  Cleaning, Shifting Left and Harmonizing...")
# all_new_labels = []
# all_count_cols = []
# all_has_cols = []
#
# for short_name, keywords in mapping_categories.items():
#     original_cols = [c for c in label_cols if any(k in c for k in keywords)]
#     if not original_cols: continue
#
#     # A. Deduplication & Left-Shift
#     df_radio[original_cols] = df_radio[original_cols].apply(clean_row_senior_priority, axis=1)
#     df_radio[original_cols] = df_radio[original_cols].apply(shift_row_left, axis=1)
#
#     # B. Harmonized renaming
#     rename_map = {old: f"{short_name}_{i+1}" for i, old in enumerate(original_cols)}
#     df_radio.rename(columns=rename_map, inplace=True)
#
#     new_cols = list(rename_map.values())
#     all_new_labels.extend(new_cols)
#
#     # C. Create flag and count columns
#     count_col_name = f'count_{short_name}'
#     has_col_name = f'has_{short_name}'
#     df_radio[has_col_name] = df_radio[new_cols].notna().any(axis=1).astype(int)
#     df_radio[count_col_name] = df_radio[new_cols].notna().sum(axis=1)
#
#     all_count_cols.append(count_col_name)
#     all_has_cols.append(has_col_name)
#
# df_radio['total_exam_count'] = df_radio[all_count_cols].sum(axis=1)
#
# # ============================================================
# # 5. MERGE WITH EMERGENCY DATA (IOA/VITAL)
# # ============================================================
# print("🔗 Merging with emergency department data...")
# df_urgences = pd.read_csv(file_urgences, dtype={'nda': str})
# df_urgences['nda'] = df_urgences['nda'].str.strip()
#
# # Prepare selection for merge
# final_selection_radio = ['nda'] + all_has_cols + all_count_cols + all_new_labels + ['total_exam_count']
# df_radio_minimal = df_radio[final_selection_radio].copy()
#
# df_final = pd.merge(df_urgences, df_radio_minimal, on='nda', how='left')
#
# # Handling Missing Values
# df_final[all_has_cols] = df_final[all_has_cols].fillna(0).astype(int)
# df_final[all_count_cols] = df_final[all_count_cols].fillna(0).astype(int)
# df_final['total_exam_count'] = df_final['total_exam_count'].fillna(0).astype(int)
# df_final['has_imaging_any'] = (df_final['total_exam_count'] > 0).astype(int)
#
# # Normalize text labels: ensure all variations of "empty" are true NaNs for the Transformer
# for col in all_new_labels:
#     df_final[col] = df_final[col].replace(['NONE', 'none', 'nan', '', 'NAT', 'NULL'], np.nan)
#
# # ============================================================
# # 6. AUTOMATIC CLEANUP: REMOVE EMPTY COLUMNS
# # ============================================================
# print("🧹 Detecting and removing entirely empty exam columns...")
#
# # Filter columns that contain 100% NaN values
# empty_cols = [col for col in all_new_labels if col in df_final.columns and df_final[col].isna().all()]
#
# if empty_cols:
#     print(f"🚫 Removing {len(empty_cols)} empty columns: {empty_cols}")
#     df_final.drop(columns=empty_cols, inplace=True)
# else:
#     print("✨ No empty columns detected.")
#


Parmis
les
6
paitents
aui
n
'ont pas d'
exam
a
pel
2022, le
1
er
c'esy un import sur le pacs, le 2eme de la pedia, le 3eme echo urgence hors imagerie, 4eme consult pluri disc poumon, le 5eme import cd pacs => donc pour moi on peut partir le fait de supprimer ces patients

In [4]:
# EXAM_MAP = {
#     # --- CT SCANS (TDM) ---
#     'TDM CRANE/CEREBRAL': 'ct brain',
#     'TDM TRAUMA CRANE': 'ct head trauma',
#     'TDM CRANE-THORAX-ABDOMEN-PELVIS': 'ct whole body trauma',
#     'TDM THORAX-ABDO-PELVIS': 'ct chest abdomen pelvis',
#     'TDM ABDO-PELVIS': 'ct abdomen pelvis',
#     'TDM ABDOMINAL': 'ct abdomen',
#     'TDM THORAX': 'ct chest',
#     'TDM ANGIOSCAN CEREBRAL': 'ct angiography brain',
#     'TDM ANGIOSCAN THORAX': 'ct angiography chest pulmonary embolism',
#     'TDM RACHIS LOMBAIRE': 'ct lumbar spine',
#     'TDM RACHIS CERVICAL': 'ct cervical spine',
#     'TDM SINUS': 'ct sinuses',
#     'TDM ROCHERS': 'ct temporal bone',
#     'TDM PERFUSION': 'ct perfusion brain',
#
#     # --- MRI (IRM) ---
#     'IRM ALERTE AVC': 'mri stroke protocol',
#     'IRM CRANE/ENCEPHALE': 'mri brain',
#     'IRM CRANE DIFF-PERF': 'mri brain diffusion perfusion',
#     'ANGIO IRM CERVICO CEREBRAL': 'mri angiography neck brain',
#     'IRM RACHIS LOMBAIRE': 'mri lumbar spine',
#     'IRM RACHIS CERVICAL': 'mri cervical spine',
#     'IRM MEDULLAIRE': 'mri spinal cord',
#     'IRM ABDOMEN': 'mri abdomen',
#     'IRM PELVIS': 'mri pelvis',
#
#     # --- X-RAY (RADIO) ---
#     'RADIO THORAX': 'xray chest',
#     'RADIO THORAX AU LIT': 'xray chest bedside',
#     'RADIO ABDOMEN SANS PREPARATION': 'xray abdomen supine',
#     'RADIO BASSIN': 'xray pelvis',
#     'RADIO RACHIS LOMBAIRE F+P': 'xray lumbar spine',
#     'RADIO EPAULE D F+P': 'xray right shoulder',
#     'RADIO EPAULE G F+P': 'xray left shoulder',
#     'RADIO GENOU D F+P': 'xray right knee',
#     'RADIO CHEVILLE D F+P': 'xray right ankle',
#     'RADIO POIGNET D F+P': 'xray right wrist',
#     'RADIO MAIN D F+ 3/4': 'xray right hand',
#
#     # --- ULTRASOUND (ECHO) ---
#     'ECHO ABDOMINALE': 'abdominal ultrasound',
#     'ECHO DOPPLER VAISSEAUX DU COU': 'carotid doppler ultrasound',
#     'ECHO DOPPLER VEINEUX MBRE INF': 'venous doppler ultrasound lower limb deep vein thrombosis',
#     'ECHO DOPPLER ARTERIEL MBRE INF': 'arterial doppler ultrasound lower limb',
#     'ECHO REINS-PELVIS': 'renal and bladder ultrasound',
#     'ECHO TESTICULAIRE': 'scrotal ultrasound',
#
#     # --- INTERVENTIONAL & SPECIAL ---
#     'THROMBECTOMIE/AVC': 'mechanical thrombectomy stroke',
#     'POSE DE PICC LINE': 'picc line insertion',
#     'POSE DE MID LINE': 'midline catheter insertion',
#     'PONCTION LOMBAIRE': 'lumbar puncture',
#     'NEPHROSTOMIE': 'percutaneous nephrostomy',
#     'DRAINAGE PELVIS': 'pelvic drainage',
#     'SCINTI POUMONS VENT/PERF': 'v/q lung scintigraphy pulmonary embolism'
# }

In [5]:
# # ============================================================
# # 7. SAVE FINAL DATASET
# # ============================================================
# df_final.to_csv(output_final_merge, index=False)
# print(f"✅ Final merged dataset saved: {output_final_merge}")
# print(f"📊 Final Shape: {df_final.shape}")